## Install `holidays`

The Databricks Serverless environment does not ship the `holidays` package. This cell installs it and restarts Python so `src.features` can import it. Only takes ~5 seconds.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


# 03 — Silver: cleaned + temporally enriched

- Drops target-leakage and post-facto columns (actual times, taxi, delay causes).
- Drops 2020 (COVID anomaly).
- Preserves `arrival_delay` nulls (cancelled / diverted); Gold filters them.
- Adds calendar + US-federal-holiday features via `src.features` (unit-tested).

Idempotent: full overwrite.

In [0]:
import sys
sys.path.append("..")

from pyspark.sql.functions import (
    col, dayofmonth, dayofweek, floor, quarter as spark_quarter,
    to_date, udf, weekofyear, when, year, month as spark_month,
)
from pyspark.sql.types import IntegerType, StringType

from src import config
from src.features import (
    check_holiday, check_holiday_period, check_near_holiday, get_season,
)

In [0]:
bronze = spark.table(config.BRONZE)
print(f"Bronze rows: {bronze.count():,}")

## Column reduction + rename

In [0]:
selected = (
    bronze
    .select(
        col("AIRLINE").alias("airline_name"),
        col("AIRLINE_CODE").alias("airline_code"),
        col("FL_NUMBER").cast("int").alias("fl_number"),
        col("ORIGIN").alias("origin_airport_code"),
        col("DEST").alias("destination_airport_code"),
        to_date(col("FL_DATE")).alias("flight_date"),
        col("CRS_DEP_TIME").cast("int").alias("crs_dep_time"),
        col("CRS_ARR_TIME").cast("int").alias("crs_arr_time"),
        col("CRS_ELAPSED_TIME").cast("double").alias("crs_elapsed_time"),
        col("DISTANCE").cast("double").alias("distance"),
        col("DEP_DELAY").cast("double").alias("dep_delay"),
        col("ARR_DELAY").cast("double").alias("arrival_delay"),
    )
)

## Filters

In [0]:
enriched = (
    selected
    .withColumn("flight_year", year(col("flight_date")))
    .filter(col("flight_year") != 2020)          # COVID anomaly
    .filter(col("flight_date").isNotNull())
)

print(f"After 2020 filter: {enriched.count():,}")

## Temporal + holiday features
Holiday flags are computed by Python UDFs backed by the unit-tested helpers in
`src.features`. Not vectorized, but only called once per row per full run.

In [0]:
season_udf = udf(get_season, StringType())
holiday_udf = udf(check_holiday, IntegerType())
near_holiday_udf = udf(check_near_holiday, IntegerType())
holiday_period_udf = udf(check_holiday_period, IntegerType())

silver = (
    enriched
    .withColumn("flight_month", spark_month(col("flight_date")))
    .withColumn("day_of_week", dayofweek(col("flight_date")))       # 1=Sunday
    .withColumn("week_of_year", weekofyear(col("flight_date")))
    .withColumn("day_of_month", dayofmonth(col("flight_date")))
    .withColumn("quarter", spark_quarter(col("flight_date")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("is_holiday", holiday_udf(col("flight_date")))
    .withColumn("is_near_holiday", near_holiday_udf(col("flight_date")))
    .withColumn("is_holiday_period", holiday_period_udf(col("flight_date")))
    .withColumn("season", season_udf(col("flight_month")))
    .withColumn("dep_hour", floor(col("crs_dep_time") / 100))
    .withColumn("arr_hour", floor(col("crs_arr_time") / 100))
)

## Write

In [0]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.SILVER)
)

silver_count = spark.table(config.SILVER).count()
print(f"Silver rows: {silver_count:,}")
print(f"Silver columns: {len(silver.columns)}")

In [0]:
%sql
DESCRIBE HISTORY workspace.flights.silver_flights;